In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [16]:
function parse_vertex(line::AbstractString)::RayTracing.Pnt3
	parts = split(strip(line))[2:end]
	if length(parts) != 3
		throw(ArgumentError("Invalid vertex format: $line"))
	end
	return RayTracing.Pnt3(parse(Float64, parts[1]), parse(Float64, parts[2]), parse(Float64, parts[3]))
end

function parse_normal(line::AbstractString)::RayTracing.Nml3
	parts = split(strip(line))[2:end]
	if length(parts) != 3
		throw(ArgumentError("Invalid vertex format: $line"))
	end
	return RayTracing.Nml3(parse(Float64, parts[1]), parse(Float64, parts[2]), parse(Float64, parts[3]))
end

function parse_uv(line::AbstractString)::RayTracing.Pnt2
	parts = split(strip(line))[2:end]
	if length(parts) != 2
		throw(ArgumentError("Invalid vertex format: $line"))
	end
	return RayTracing.Pnt2(parse(Float64, parts[1]), parse(Float64, parts[2]))
end

function parse_face!(
    line::AbstractString,
    vertex_indices::Vector{Int},
    uv_indices::Vector{Int},
    normal_indices::Vector{Int}
)
	parts = split(strip(line))[2:end]
    
    if length(parts) != 3
        throw(ArgumentError("Invalid face format on line: $line"))
    end
	
	for part in parts
		indices = split(part, "/")
		push!(vertex_indices, parse(Int, indices[1]))
		
        if length(indices) > 1 && indices[2] != ""
            push!(uv_indices, parse(Int, indices[2]))
        end
		
		if length(indices) > 2 && indices[3] != ""
            push!(normal_indices, parse(Int, indices[3]))
        end
    end
end

function parse_obj2(
    file_path::AbstractString,
    object_to_world::RayTracing.Transformation, 
    reverse_orientation::Bool, 
    transform_swaps_handedness::Bool,
    alpha_mask::RayTracing.Maybe{RayTracing.Texture},
)
	vertices = RayTracing.Pnt3[]
    vertex_indices = Int[]

	uvs = RayTracing.Pnt2[]
    uv_indices = Int[]

	normals = RayTracing.Nml3[]
	normal_indices = Int[]
	
	open(file_path) do file
		for line in eachline(file)
			line = strip(line)
			if isempty(line) || startswith(line, "#")
				continue
			end
			
			parts = split(line)
			cmd = parts[1]
			
            if cmd == "v"
                push!(vertices, parse_vertex(line))
            elseif cmd == "vt"
                push!(uvs, parse_uv(line))
            elseif cmd == "vn"
                push!(normals, parse_normal(line))
            elseif cmd == "f"
                parse_face!(line, vertex_indices, uv_indices, normal_indices)
            elseif cmd == "mtllib"
                @warn "Skipping material: $line"
                continue
            elseif cmd == "usemtl"
                @warn "Skipping material: $line"
                continue
            elseif cmd == "g"
                @warn "Skipping group: $line"
                continue
            else
                @assert false
            end
		end
	end

    # if face only specifies a single set of indices but we get normals
    if (length(normals) > 0) & (length(normal_indices) == 0)
        @assert length(vertices) == length(normals)
        normal_indices = vertex_indices
    end

    # if face only specifies a single set of indices but we get uvs
    if (length(uvs) > 0) & (length(uv_indices) == 0)
        @assert length(vertices) == length(uvs)
        uv_indices = vertex_indices
    end

    @assert mod(length(vertex_indices), 3) == 0
    @assert mod(length(uv_indices), 3) == 0
    @assert mod(length(normal_indices), 3) == 0

    normals = length(normals)>0 ? normals : nothing
    normal_indices = length(normal_indices)>0 ? normal_indices : nothing

    uvs = length(uvs)>0 ? uvs : nothing
    uv_indices = length(uv_indices)>0 ? uv_indices : nothing

	return RayTracing.construct_triangle_mesh(
        RayTracing.ShapeCore(object_to_world, RayTracing.Inv(object_to_world), reverse_orientation, transform_swaps_handedness),
        length(vertex_indices)÷3,
    
        vertices,
        vertex_indices,
    
        normals,
        normal_indices,
    
        uvs,
        uv_indices,
    
        alpha_mask
    )
end


parse_obj2 (generic function with 1 method)

In [17]:
tris = parse_obj2(
    RayTracing.jmfp("/home/jmyslinski/random_stuff/PBRJ/test/plane.obj"),
    RayTracing.Translate(RayTracing.Pnt3(0,0,0)),
    false,
    false,
    nothing
)

┌ Warning: Skipping material: mtllib plane.mtl
└ @ Main /home/jmyslinski/random_stuff/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sdnNjb2RlLXJlbW90ZQ==.jl:86
┌ Warning: Skipping material: usemtl Material
└ @ Main /home/jmyslinski/random_stuff/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sdnNjb2RlLXJlbW90ZQ==.jl:89


2-element Vector{Main.RayTracing.Triangle}:
 Main.RayTracing.Triangle(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), false, false), Main.RayTracing.TriangleMesh(2, Main.RayTracing.Pnt3[[-1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]], [1, 2, 3, 1, 2, 4], Main.RayTracing.Nml3[[0.0, 1.0, 0.0]], [1, 1, 1, 1, 1, 1], Main.RayTracing.Pnt2[[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]], [1, 2, 3, 1, 2, 4], nothing, nothing), 1)
 Main.RayTracing.Triangle(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main

In [18]:
o = RayTracing.Pnt3(5,5,5)
p = RayTracing.Pnt3(.5, 0, .5)
ray1 = RayTracing.Ray(
    o,
    RayTracing.Vec3(p - o),
    0.0,
    typemax(Float64)
)

[5.0, 5.0, 5.0], [-4.5, -5.0, -4.5], 0.0, Inf, false

In [19]:
@assert RayTracing.intersect_p(tris[1], ray1)
@assert !RayTracing.intersect_p(tris[2], ray1)

In [20]:
tris = parse_obj2(
    RayTracing.jmfp("/home/jmyslinski/random_stuff/PBRJ/ref/teapot.obj"),
    RayTracing.Translate(RayTracing.Pnt3(0,0,0)),
    false,
    false,
    nothing
)

┌ Warning: Skipping group: g Object001
└ @ Main /home/jmyslinski/random_stuff/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sdnNjb2RlLXJlbW90ZQ==.jl:92


1024-element Vector{Main.RayTracing.Triangle}:
 Main.RayTracing.Triangle(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), false, false), Main.RayTracing.TriangleMesh(1024, Main.RayTracing.Pnt3[[40.6266, 28.3457, -1.10804], [40.0714, 30.4443, -1.10804], [40.7155, 31.1438, -1.10804], [42.0257, 30.4443, -1.10804], [43.4692, 28.3457, -1.10804], [37.5425, 28.3457, 14.5117], [37.0303, 30.4443, 14.2938], [37.6244, 31.1438, 14.5466], [38.8331, 30.4443, 15.0609], [40.1647, 28.3457, 15.6274]  …  [13.7313, 30.8773, -31.4277], [15.3351, 28.3457, -35.1972], [10.0391, 34.3417, -10.3161], [17.4812, 32.6095, -17.7582], [24.1665, 30.8773, -24.4435], [27.0677, 28.3457, -27.3447], [12.795,

In [21]:
minimum(tris[1].mesh.vertex_indices)

1

In [22]:
minimum(tris[1].mesh.normal_indices)

1

In [23]:
minimum(tris[1].mesh.uv_indices, init=typemax(Float64))

MethodError: MethodError: no method matching iterate(::Nothing)

Closest candidates are:
  iterate(!Matched::LazyString)
   @ Base strings/lazy.jl:94
  iterate(!Matched::LazyString, !Matched::Integer)
   @ Base strings/lazy.jl:95
  iterate(!Matched::Core.Compiler.InstructionStream, !Matched::Int64)
   @ Base show.jl:2778
  ...
